In [1]:
import aiohttp
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
import json
import logging
import os

# Create folders for results and errors
os.makedirs("result_20k", exist_ok=True)
os.makedirs("error_20k", exist_ok=True)

# Configure logging
logging.basicConfig(
    filename="log-20k.txt",
    level=logging.INFO,
    format="%(message)s",
    encoding="utf-8"
)

semaphore = asyncio.Semaphore(40)  #

async def fetch(session, url, idx, product_id):
    async with semaphore:
        try:
            async with session.get(url) as response:
                if response.status == 200:
                    try:
                        data = await response.json()
                    except Exception:
                        msg = f"[ERROR] index={idx}, id={product_id}, error=Can not parse JSON"
                        print(msg)
                        logging.error(msg)
                        return None, msg

                    raw_description = data.get("description", "")
                    soup = BeautifulSoup(raw_description, "html.parser")
                    clean_description = soup.get_text(separator=" ", strip=True)

                    product_info = {
                        "id": data.get("id"),
                        "name": data.get("name"),
                        "url_key": data.get("url_key"),
                        "price": data.get("price"),
                        "description": clean_description,
                        "images": [img.get("base_url") for img in data.get("images", [])]
                    }

                    msg = f"[OK] index={idx}, id={product_id}"
                    print(msg)
                    logging.info(msg)
                    return product_info, None
                else:
                    msg = f"[ERROR] index={idx}, id={product_id}, error=HTTP {response.status}"
                    print(msg)
                    logging.error(msg)
                    return None, msg
        except Exception as e:
            msg = f"[ERROR] index={idx}, id={product_id}, error={str(e)}"
            print(msg)
            logging.error(msg)
            return None, msg

async def process_batch(batch_df, batch_index):
    products = []
    error_msgs = []

    async with aiohttp.ClientSession(headers={"User-Agent": "Mozilla/5.0"}) as session:
        tasks = []
        for idx, product_id in enumerate(batch_df["id"]):
            url = f"https://api.tiki.vn/product-detail/api/v1/products/{product_id}"
            tasks.append(fetch(session, url, idx + batch_index * len(batch_df), product_id))

        results = await asyncio.gather(*tasks)

    for result, err in results:
        if result:
            products.append(result)
        if err:
            error_msgs.append(err)

    # Save batch result json file
    with open(f"result_20k/product_batch_{batch_index+1}.json", "w", encoding="utf-8") as f:
        json.dump({"data": products}, f, ensure_ascii=False, indent=4)

    # Save batch error file
    with open(f"error_20k/errors_batch_{batch_index+1}.txt", "w", encoding="utf-8") as f:
        for msg in error_msgs:
            f.write(msg + "\n")

    summary = (
        f"=== Batch {batch_index+1} ===\n"
        f"Numbers of products GET successfully: {len(products)}\n"
        f"Numbers of ERROR: {len(error_msgs)}\n"
        f"Saved to result_20k/product_batch_{batch_index+1}.json\n"
        f"Saved to error_20k/errors_batch_{batch_index+1}.txt\n"
    )
    print(summary)
    logging.info(summary)

async def main():
    df = pd.read_csv("products-0-200000.csv")
    batch_size = 1000
    num_batches = (len(df) + batch_size - 1) // batch_size

    for batch_index in range(num_batches):
        start = batch_index * batch_size
        end = min((batch_index + 1) * batch_size, len(df))
        batch_df = df.iloc[start:end]

        print(f"=== Processing batch {batch_index+1}/{num_batches}, including {len(batch_df)} products ===")
        await process_batch(batch_df, batch_index)

if __name__ == "__main__":
    await main()


=== Processing batch 1/200, including 1000 products ===
[OK] index=35, id=36376636
[OK] index=17, id=191463903
[OK] index=23, id=215101707
[OK] index=14, id=139457787
[OK] index=2, id=154155413
[OK] index=36, id=140420532
[OK] index=15, id=146457002
[OK] index=16, id=178596691
[OK] index=38, id=138838287
[OK] index=1, id=74897599
[OK] index=33, id=160814241
[OK] index=31, id=168663152
[OK] index=0, id=1391347
[OK] index=28, id=67029245
[OK] index=5, id=214009046
[OK] index=41, id=179805786
[OK] index=19, id=170337343
[OK] index=29, id=154530234
[OK] index=6, id=171618108
[OK] index=27, id=103637219
[OK] index=18, id=167750518
[OK] index=13, id=75331097
[OK] index=8, id=139457837
[OK] index=3, id=253117062
[OK] index=42, id=213678732
[OK] index=40, id=164361837
[OK] index=30, id=157943579
[OK] index=44, id=75378750
[OK] index=45, id=213678772
[OK] index=37, id=206303845
[OK] index=12, id=138083218
[OK] index=43, id=174287389
[OK] index=9, id=197334787
[OK] index=46, id=176464521
[OK] in